# SNOMED CT Term-to-Code & Concept Expansion Framework

This notebook demonstrates multi-strategy medical term expansion. Given an initial input list of terms (starting with `['meningioma']`), this workflow applies four distinct expansion mechanisms:

1. **Lexical & Prefix Lookup**: Uses exact, prefix, and fuzzy matching over SNOMED CT description files.
2. **Hierarchical Tree Expansion**: Traverses parent (`sourceId`) and child (`destinationId`) relationships across SNOMED CT RF2 stated/inferred graph structures.
3. **MedCAT CDB Similarity**: Utilizes context-vector embeddings learned from clinical natural language processing to retrieve semantically related CUIs.
4. **GatorTron Embedding Cosine Similarity**: Uses deep clinical transformer embeddings to capture subtle semantic relatedness in high-dimensional vector space.

## Step 1: Global Setup & Path Configuration

In [ ]:
import glob
import os
import pickle
import sys
from typing import Dict, Set, Tuple

import pandas as pd
from IPython.display import display  # noqa: A004

from snomed_methods_v1 import SnomedRelations
from snomed_term_lookup import create_term_lookup_from_directory

# Ensure project root is in Python path,
# adjusting relative to notebook execution location
CURRENT_DIR = (
    os.path.dirname(os.path.abspath(__file__))
    if "__file__" in locals()
    else os.getcwd()
)
PROJECT_ROOT = (
    os.path.abspath(os.path.join(CURRENT_DIR, ".."))
    if CURRENT_DIR.endswith("notebooks")
    else "/workspaces/snomed_methods"
)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Resolve modelpack default dynamically (.zip file in model_packs folder)
MODEL_PACKS_DIR = os.path.join(PROJECT_ROOT, "model_packs")
zip_modelpacks = glob.glob(os.path.join(MODEL_PACKS_DIR, "*.zip"))
DEFAULT_MEDCAT_MODEL = (
    zip_modelpacks[0]
    if zip_modelpacks
    else os.path.join(MODEL_PACKS_DIR, "modelpack.zip")
)

# Configuration Parameters
MEDCAT_MODEL_PATH = os.environ.get("MEDCAT_MODEL_PATH", DEFAULT_MEDCAT_MODEL)
UK_SNOMED_DIR = os.environ.get(
    "UK_SNOMED_DIR",
    os.path.join(
        PROJECT_ROOT,
        "uk_sct2cl_42.2.0/SnomedCT_UKClinicalRF2_PRODUCTION_20260603T000001Z",
    ),
)
STATED_REL_PATH = os.environ.get(
    "SCT2_PATH",
    os.path.join(
        PROJECT_ROOT,
        "uk_sct2cl_42.2.0/SnomedCT_InternationalRF2_PRODUCTION_20260201T120000Z/Full/Terminology/sct2_StatedRelationship_Full_INT_20260201.txt",
    ),
)
GATORTRON_EMBEDDINGS_PATH = os.environ.get(
    "GATORTRON_EMBEDDING_PATH",
    os.path.join(PROJECT_ROOT, "uk_sct2cl_42.2.0/gatortron_embeddings.pkl"),
)

# Input seed terms specified as a list
INPUT_TERMS = ["meningioma"]

## Step 2: Method 1 - Direct Lexical & Prefix Matching

**How it works:** Queries the `SnomedTermLookup` engine for exact term matches, prefix matches (e.g., matching stems like `meningiom-`), and token substring occurrences across all official SNOMED CT Fully Specified Names (FSN) and Synonyms.

In [ ]:
lookup = create_term_lookup_from_directory(UK_SNOMED_DIR)

lexical_results: Set[Tuple[str, str]] = set()
for term in INPUT_TERMS:
    # Direct term matching
    matches = lookup.find_concepts_by_term(term, top_n=100)
    for cui, matched_term in matches:
        lexical_results.add((str(cui), matched_term))

    # Prefix matching for variants
    prefix_matches = lookup.find_concepts_by_term(term, match_prefix=True, top_n=50)
    for cui, matched_term in prefix_matches:
        lexical_results.add((str(cui), matched_term))


## Step 3: Method 2 - Hierarchical Graph Traversal (Parents & Children)

**How it works:** Seeds the recursive relationship expander with CUIs identified in Step 1. It traverses the `is_a` (116680003) relationships up (parents/ancestors) and down (children/descendants) the hierarchy up to a configured recursion depth ($N$).

In [ ]:
snomed_rel = SnomedRelations(
    snomed_rf2_full_path=STATED_REL_PATH,
    medcat_path=MEDCAT_MODEL_PATH,
    medcat=True,
    dhcap02=True,
)

seed_cuis = list({cui for cui, _ in lexical_results})
hierarchical_results: Set[Tuple[str, str]] = set()

# Expand top seed concepts to depth 2
for cui in seed_cuis[:5]:
    codes, names = snomed_rel.recursive_code_expansion(cui, n_recursion=2, debug=False)
    for c, n in zip(codes, names):
        pref_name = (
            n
            if n is not None
            else lookup.getconcept_info(str(c)).get("preferred_name", "N/A")
        )
        hierarchical_results.add((str(c), pref_name))


## Step 4: Method 3 - MedCAT Concept Database Vector Similarity

**How it works:** Leverages MedCAT's pre-trained Concept Database (`cdb`) context embeddings (`xxxlong` window). It calculates cosine similarities between the context vector of the seed concept and all indexed clinical concepts to surface semantically aligned terms regardless of hierarchy placement.

In [ ]:
medcat_results: Set[Tuple[str, str]] = set()

if snomed_rel.has_medcat():
    for cui in seed_cuis[:3]:
        codes, names = snomed_rel.get_medcat_cdb_most_similar(
            cui, context_type="xxxlong", topn=20
        )
        for c, n in zip(codes, names):
            medcat_results.add((str(c), str(n)))
else:
    pass

## Step 5: Method 4 - Deep Transformer (GatorTron) Vector Space Search

**How it works:** Loads dense clinical language representation vectors (GatorTron). Computes high-dimensional cosine similarity between the mean vector of seed terms and candidate concept embeddings to uncover implicitly related conditions.

In [ ]:
gatortron_results: Set[Tuple[str, str]] = set()

if os.path.exists(GATORTRON_EMBEDDINGS_PATH):
    from sklearn.metrics.pairwise import cosine_similarity

    with open(GATORTRON_EMBEDDINGS_PATH, "rb") as f:
        embeddings_dict = pickle.load(f)

    seed_term = INPUT_TERMS[0].lower()
    if seed_term in embeddings_dict:
        target_vec = embeddings_dict[seed_term].reshape(1, -1)
        sims = {}
        for term, vec in list(embeddings_dict.items())[
            :5000
        ]:  # Sample sub-space for performance
            sims[term] = cosine_similarity(target_vec, vec.reshape(1, -1))[0, 0]

        top_terms = sorted(sims.items(), key=lambda x: x[1], reverse=True)[1:21]
        for term, _score in top_terms:
            matches = lookup.find_concepts_by_term(term, top_n=1)
            if matches:
                gatortron_results.add((str(matches[0][0]), matches[0][1]))
else:
    pass

## Step 6: Combined Results Aggregation & Method Metrics

Consolidates candidate concepts across all methods, deduplicates by CUI, and summarizes the yield metrics per strategy.

In [ ]:
# Aggregation Map: CUI -> {preferred_name, sources}
aggregated_concepts: Dict[str, Dict] = {}


def register_results(results_set, source_name):
    for cui, term in results_set:
        cui_str = str(cui)
        if cui_str not in aggregated_concepts:
            aggregated_concepts[cui_str] = {"preferred_name": term, "sources": set()}
        aggregated_concepts[cui_str]["sources"].add(source_name)


register_results(lexical_results, "Lexical/Prefix")
register_results(hierarchical_results, "Hierarchical Tree")
register_results(medcat_results, "MedCAT CDB")
register_results(gatortron_results, "GatorTron Vector")

# Construct DataFrame Summary
summary_data = []
for cui, details in aggregated_concepts.items():
    summary_data.append(
        {
            "SNOMED_CUI": cui,
            "Concept_Name": details["preferred_name"],
            "Found_By_Count": len(details["sources"]),
            "Discovery_Methods": ", ".join(sorted(details["sources"])),
        }
    )

df_results = pd.DataFrame(summary_data).sort_values(
    by=["Found_By_Count", "SNOMED_CUI"], ascending=[False, True]
)

# Print Discovery Metrics

# Display preview table

display(df_results.head(25))

In [ ]:
# Export full term expansion mapping to CSV
output_csv = os.path.join(
    PROJECT_ROOT, "notebooks", "snomed_meningioma_expansion_results.csv"
)
df_results.to_csv(output_csv, index=False)